# 📊 Final Project: Exploring the Impact of Socioeconomic Factors on Crime Rates in the U.S. (2010–2019)

## US Crime, Employment & Income Analysis (2010–2019)

This Jupyter Notebook is part of a data visualization and analysis project exploring the relationship between crime rates, unemployment, and income levels across U.S. states over the years 2010–2019.

### 📁 What's Included:
- **Crime Data** from the FBI's Uniform Crime Reporting (UCR) Program.
- **Employment Status** data from the U.S. Census Bureau's ACS S2301 tables.
- **Income Distribution** data from the U.S. Census Bureau's ACS S1901 tables.

### 🔄 Processing Steps:
1. **Unzip and extract** source files from downloaded ZIPs.
2. **Convert** `.xls` to `.xlsx` where necessary.
3. **Clean & filter** each dataset to retain relevant columns.
4. **Combine yearly files** into clean, state-level datasets for:
   - Crime  
   - Employment  
   - Income
5. **Save outputs** into separate cleaned `.csv` files for future analysis.

### 🎯 Next Steps:
- Visualize trends and correlations.
- Explore questions such as:
  - Does income inequality influence property crime rates (e.g., burglary, larceny-theft) across U.S. states?
  - Are states with lower income levels experiencing more property cimes?

Let’s dive in 🚀
🚀


## 📦 Step 0: Install Required Python Packages

Before processing any data, install the required libraries:
- `pandas`, `openpyxl`: for handling Excel/CSV files
- `pyexcel`, `pyexcel-xls`, `pyexcel-xlsx`: for converting `.xls` to `.xlsx` (older Excel formats)


In [6]:
# Install core libraries for Excel and CSV handling
!pip install pandas openpyxl

# Install pyexcel plugins to convert .xls → .xlsx (for legacy Excel files)
!pip install pyexcel pyexcel-xls pyexcel-xlsx


In [7]:
#Import necessary libraries
import os          # For handling file paths and directories
import zipfile     # To extract files from ZIP archives
import pandas as pd  # For data manipulation and analysis

## 📂 Step 1: Unzip FBI Crime Data Files

Unzip the original FBI crime dataset ZIP file.  
This step extracts all `.xls` files (one per year from 2010–2019) into a designated folder.

This version ensures folders inside the ZIP (if any) are handled properly by checking each entry.

*Note*: Original FBI Crime data was provided in `.xls` format.  
For compatibility and reproducibility, files were converted to `.xlsx` using Excel automation.


In [11]:
# Define the ZIP file location and extraction folder
crime_zip_path = r"C:/Users/mayan/Downloads/DJ/Study/USF/Spring 2025/Data Visualization/Final Project/Project/Crime Data 2010-2019.zip"
crime_extract_to = r"C:/Users/mayan/Downloads/DJ/Study/USF/Spring 2025/Data Visualization/Final Project/Project/Crime Data 2010-2019"

# Create output folder if it doesn't already exist
os.makedirs(crime_extract_to, exist_ok=True)

# Unzip the contents, skipping over folder entries
with zipfile.ZipFile(crime_zip_path, 'r') as crime_zip_ref:
    for member in crime_zip_ref.infolist():
        if not member.is_dir():  # Only extract files, not subfolders
            member_path = os.path.join(crime_extract_to, member.filename)
            os.makedirs(os.path.dirname(member_path), exist_ok=True)
            with open(member_path, 'wb') as f:
                f.write(crime_zip_ref.read(member))

print("ZIP extracted with folders handled correctly")


ZIP extracted with folders handled correctly


## 🔄 Step 2: Convert `.xls` Files to `.xlsx`

To ensure compatibility with `pandas` and modern Excel tools, we convert all FBI crime files from `.xls` (older Excel format) to `.xlsx` using `pyexcel`.

- Handles each yearly file
- Skips temporary lock files (`~$`)
- Preserves original content and structure


In [14]:
import pyexcel as pe

# Folder containing all the .xls crime data files
xls_folder = os.path.join(crime_extract_to, "Crime Data 2010-2019")

# Loop through all files in the folder
for filename in os.listdir(xls_folder):
    # Convert only .xls files, skip temporary Excel lock files
    if filename.endswith(".xls") and not filename.startswith("~$"):
        xls_path = os.path.join(xls_folder, filename) #Build the full path to the original .xls file.
        xlsx_path = xls_path.replace(".xls", ".xlsx") #Create the corresponding output file path with .xlsx instead of .xls
        print(f"Converting {filename} → {os.path.basename(xlsx_path)}")

        try:
            # Read the .xls sheet and save as .xlsx
            sheet = pe.get_sheet(file_name=xls_path) #This reads the .xls file into memory using pyexcel
            sheet.save_as(xlsx_path) #This saves the previously opened sheet to a new file, now in .xlsx format
            print(f"Converted: {os.path.basename(xlsx_path)}")
        except Exception as e:
            print(f"Failed: {filename} — {e}")


Converting 2010.xls → 2010.xlsx
Converted: 2010.xlsx
Converting 2011.xls → 2011.xlsx
Converted: 2011.xlsx
Converting 2012.xls → 2012.xlsx
Converted: 2012.xlsx
Converting 2013.xls → 2013.xlsx
Converted: 2013.xlsx
Converting 2014.xls → 2014.xlsx
Converted: 2014.xlsx
Converting 2015.xls → 2015.xlsx
Converted: 2015.xlsx
Converting 2016.xls → 2016.xlsx
Converted: 2016.xlsx
Converting 2017.xls → 2017.xlsx
Converted: 2017.xlsx
Converting 2018.xls → 2018.xlsx
Converted: 2018.xlsx
Converting 2019.xls → 2019.xlsx
Converted: 2019.xlsx


## 📦 STEP 3: Clean and Combine FBI Crime Data (2010–2019)
In this step, standardized crime data for each U.S. state from 2010 to 2019 were extracted.
Each Excel file contains crime statistics for a specific year. The task is to pull only the state-level total crimes and organize it into a clean, consistent format with the following fields:

* ```state, year, population,```

* ```violent_crime, murder, rape, robbery, aggravated_assault,```

* ```property_crime, burglary, larceny_theft, motor_vehicle_theft```

Only one row per state per year was retained — the ```"State Total"``` row — allowing for easy year-over-year and cross-state comparisons.

In [17]:
#Path to folder containing all yearly Excel files (already converted to .xlsx)
folder_path = xls_folder
all_data = []

# Define official 50 U.S. states (uppercase for matching)
us_states = [
    "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut", "Delaware",
    "Florida", "Georgia", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa", "Kansas", "Kentucky",
    "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota", "Mississippi",
    "Missouri", "Montana", "Nebraska", "Nevada", "New Hampshire", "New Jersey", "New Mexico",
    "New York", "North Carolina", "North Dakota", "Ohio", "Oklahoma", "Oregon", "Pennsylvania",
    "Rhode Island", "South Carolina", "South Dakota", "Tennessee", "Texas", "Utah", "Vermont",
    "Virginia", "Washington", "West Virginia", "Wisconsin", "Wyoming"
]
us_states_upper = [s.upper() for s in us_states]

# Loop through yearly files (2010–2019)
for year in range(2010, 2020):
    file_path = os.path.join(folder_path, f"{year}.xlsx") # Construct the full file path for each year's Excel file (e.g., 2013.xlsx)
    try:
        print(f"Reading: {file_path}")
        df = pd.read_excel(file_path, skiprows=3) # Read the Excel file while skipping the top 3 metadata rows
        last_state = None  # Track the last seen state name in the file

        for i, row in df.iterrows():
            # Capture the most recent non-empty state name from the first column
            if isinstance(row.iloc[0], str) and row.iloc[0].strip() != "":
                last_state = row.iloc[0].strip()

            # If it's the 'State Total' row, extract and store crime values for the current state and year
            if isinstance(row.iloc[1], str) and "State Total" in row.iloc[1]:
                values = row.values
                # Create a dictionary 'new_row' with state, year, and crime stats from the 'State Total' entry
                new_row = { 
                    "state": last_state,
                    "year": year,
                    "population": values[3],
                    "violent_crime": values[4],
                    "murder": values[5],
                    "rape": values[6],
                    "robbery": values[7],
                    "aggravated_assault": values[8],
                    "property_crime": values[9],
                    "burglary": values[10],
                    "larceny_theft": values[11],
                    "motor_vehicle_theft": values[12],
                }
                all_data.append(new_row) #Add dictionary to 'all_data', which collects cleaned rows for all states and years

    except Exception as e:
         # Skip files with issues and print the error
        print(f"Skipping {file_path} due to: {e}")

# Save to final .csv
if all_data:
    final_df = pd.DataFrame(all_data)
    # Clean state names (remove numbers, punctuation, extra whitespace)
    final_df["state"] = (
        final_df["state"]
        .str.replace(r'[^A-Za-z\s]', '', regex=True)
        .str.replace(r'\d+', '', regex=True)
        .str.strip()
        .str.upper()
    )
    final_df = final_df[final_df["state"].isin(us_states_upper)]

    #Normalize crime counts per 100,000 population
    rate_columns = [
            'violent_crime', 'murder', 'rape', 'robbery', 'aggravated_assault',
            'property_crime', 'burglary', 'larceny_theft', 'motor_vehicle_theft'
                    ]
    for col in rate_columns:
        final_df[f"{col}_rate"] = (final_df[col] / final_df['population']) * 100000
    
    output_file = os.path.join(folder_path, "FBI_Crime_Cleaned_2010_2019.csv") # Define the output file path by joining the folder path with the desired filename
    final_df.to_csv(output_file, index=False) # Save the cleaned DataFrame to a CSV file at the specified location without the index column
    print(f"\n Cleaned crime data saved to: {output_file}")
else:
    print("\n No valid data processed.")


Reading: C:/Users/mayan/Downloads/DJ/Study/USF/Spring 2025/Data Visualization/Final Project/Project/Crime Data 2010-2019\Crime Data 2010-2019\2010.xlsx
Reading: C:/Users/mayan/Downloads/DJ/Study/USF/Spring 2025/Data Visualization/Final Project/Project/Crime Data 2010-2019\Crime Data 2010-2019\2011.xlsx
Reading: C:/Users/mayan/Downloads/DJ/Study/USF/Spring 2025/Data Visualization/Final Project/Project/Crime Data 2010-2019\Crime Data 2010-2019\2012.xlsx
Reading: C:/Users/mayan/Downloads/DJ/Study/USF/Spring 2025/Data Visualization/Final Project/Project/Crime Data 2010-2019\Crime Data 2010-2019\2013.xlsx
Reading: C:/Users/mayan/Downloads/DJ/Study/USF/Spring 2025/Data Visualization/Final Project/Project/Crime Data 2010-2019\Crime Data 2010-2019\2014.xlsx
Reading: C:/Users/mayan/Downloads/DJ/Study/USF/Spring 2025/Data Visualization/Final Project/Project/Crime Data 2010-2019\Crime Data 2010-2019\2015.xlsx
Reading: C:/Users/mayan/Downloads/DJ/Study/USF/Spring 2025/Data Visualization/Final Proj

In [19]:
# Load the cleaned crime dataset (2010–2019) for preview
crime_df = pd.read_csv(r"C:/Users/mayan/Downloads/DJ/Study/USF/Spring 2025/Data Visualization/Final Project/Project/Crime Data 2010-2019/Crime Data 2010-2019\FBI_Crime_Cleaned_2010_2019.csv")
crime_df


,state,year,population,violent_crime,murder,rape,robbery,aggravated_assault,property_crime,burglary,...,motor_vehicle_theft,violent_crime_rate,murder_rate,rape_rate,robbery_rate,aggravated_assault_rate,property_crime_rate,burglary_rate,larceny_theft_rate,motor_vehicle_theft_rate
0,ALABAMA,2010,4779736.0,18056,273.0,1349.0,4761.0,11673.0,168092,42034.0,...,10600.0,377.761450,5.711613,28.223316,99.608012,244.218509,3516.763269,879.420955,2415.572743,221.769571
1,ALASKA,2010,710231.0,4537,31.0,533.0,594.0,3379.0,20259,3105.0,...,1619.0,638.806248,4.364777,75.046006,83.634761,475.760703,2852.452230,437.181706,2187.316521,227.954004
2,ARIZONA,2010,6392017.0,26085,409.0,2165.0,6937.0,16574.0,225893,50771.0,...,21508.0,408.087150,6.398606,33.870373,108.525994,259.292176,3533.986221,794.287625,2403.216387,336.482209
3,ARKANSAS,2010,2915918.0,14735,138.0,1312.0,2372.0,10913.0,103775,32511.0,...,5544.0,505.329711,4.732643,44.994407,81.346595,374.256066,3558.913522,1114.949049,2253.835670,190.128803
4,CALIFORNIA,2010,37253956.0,164133,1809.0,8331.0,58116.0,95877.0,981939,228857.0,...,152524.0,440.578713,4.855860,22.362726,155.999540,257.360587,2635.797927,614.315967,1612.065038,409.416922
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,VIRGINIA,2019,8535519.0,17753,426.0,2816.0,3524.0,10987.0,140213,13900.0,...,10269.0,207.989696,4.990909,32.991550,41.286300,128.720937,1642.700344,162.848914,1359.542402,120.309029
496,WASHINGTON,2019,7614893.0,22377,198.0,3332.0,5147.0,13700.0,204224,34540.0,...,24402.0,293.858364,2.600168,43.756360,67.591232,179.910604,2681.902425,453.584837,1907.866598,320.450990
497,WEST VIRGINIA,2019,1792147.0,5674,78.0,754.0,378.0,4464.0,28376,5891.0,...,2419.0,316.603493,4.352322,42.072442,21.092020,249.086710,1583.352258,328.711875,1119.662617,134.977767
498,WISCONSIN,2019,5822434.0,17070,175.0,2261.0,2991.0,11643.0,85672,12667.0,...,7385.0,293.176359,3.005616,38.832557,51.370269,199.967917,1471.412128,217.555064,1127.020074,126.836989


## ✅ Crime Data (2010–2019) Preview Summary
The cleaned dataset now contains:

📍 **500 rows – one per state per year (50 states × 10 years).**

📊 **21 columns capturing essential crime metrics:**

* `population`

* `violent_crime, property_crime`

* `murder, rape, robbery, aggravated_assault`

* `burglary, larceny_theft, motor_vehicle_theft`

This dataset is now clean, consistent, and ready for analysis and visualizations.

## 📦 Step 4: Extract Employment Status Data (2010–2019)
The original ZIP file for employment data includes multiple files per year (metadata, notes, raw data). However, only the actual data files that contain employment status numbers are needed.

This step:

* Extracts only the files that end with `*-Data.csv`

* Skips metadata and notes

* Outputs them to the `Employment Status` folder for cleaning and combination

In [23]:
# Define paths
emp_zip_path = r"C:/Users/mayan/Downloads/DJ/Study/USF/Spring 2025/Data Visualization/Final Project/Project/S2301 Employment Status.zip"
emp_extract_path = r"C:/Users/mayan/Downloads/DJ/Study/USF/Spring 2025/Data Visualization/Final Project/Project/Employment Status"

# Create target directory if not exists
os.makedirs(emp_extract_path, exist_ok=True)

# Selective extraction
with zipfile.ZipFile(emp_zip_path, 'r') as emp_zip_ref:
    # Filter only files that end with -Data.csv
    data_files = [f for f in emp_zip_ref.namelist() if f.endswith("-Data.csv")]
    
    for file in data_files:
        emp_zip_ref.extract(file, emp_extract_path) # Extract the filtered data files

print("Only *-Data.csv files extracted successfully!")


Only *-Data.csv files extracted successfully!


## 🧮 Step 5: Clean & Combine Employment Status Data (2010–2019)

This step processes the U.S. Census Bureau’s ACS **S2301 Employment Status** data for the years **2010 to 2019**.

Each file includes labor force participation and unemployment statistics segmented by:

- 👨‍👩‍👧 **Gender** (male/female)  
- 📅 **Age groups** (e.g., 16–19, 20–24, ..., 75+)  
- 🌎 **Race & Ethnicity** (White, Black, Hispanic, Asian, etc.)

---

💡 **What this script does:**

- Selects only `*-Data.csv` files from the extracted ZIP  
- Keeps only relevant columns for analysis  
- Renames columns for clarity and readability  
- Cleans numeric values (e.g., removes commas, handles missing entries)  
- Filters only the **50 U.S. states** (excludes territories & D.C.)  
- Merges all years into a single, tidy CSV file for analysis


In [26]:
#Path where all -Data.csv files are located
folder_path = emp_extract_path

#List all yearly files
files = [f for f in os.listdir(folder_path) if f.endswith("-Data.csv")]

#List of 50 U.S. states
us_states = [
    "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut", "Delaware",
    "Florida", "Georgia", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa", "Kansas", "Kentucky",
    "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota", "Mississippi",
    "Missouri", "Montana", "Nebraska", "Nevada", "New Hampshire", "New Jersey", "New Mexico",
    "New York", "North Carolina", "North Dakota", "Ohio", "Oklahoma", "Oregon", "Pennsylvania",
    "Rhode Island", "South Carolina", "South Dakota", "Tennessee", "Texas", "Utah", "Vermont",
    "Virginia", "Washington", "West Virginia", "Wisconsin", "Wyoming"
]
us_states_upper = [s.upper() for s in us_states]


#Columns to keep with mapping to clean names
columns_map = {
    "NAME": "state",
    "S2301_C01_001E": "population_16_plus",
    "S2301_C02_001E": "in_labor_force_16_plus",
    "S2301_C03_001E": "employed_16_plus",
    "S2301_C04_001E": "unemployment_rate_total",
    "S2301_C04_020E": "unemployment_rate_male",
    "S2301_C04_021E": "unemployment_rate_female",
    "S2301_C04_002E": "unemployment_rate_age_16_19",
    "S2301_C04_003E": "unemployment_rate_age_20_24",
    "S2301_C04_004E": "unemployment_rate_age_25_44",
    "S2301_C04_005E": "unemployment_rate_age_45_54",
    "S2301_C04_006E": "unemployment_rate_age_55_64",
    "S2301_C04_007E": "unemployment_rate_age_65_74",
    "S2301_C04_008E": "unemployment_rate_age_75_plus",
    "S2301_C04_010E": "unemployment_rate_white",
    "S2301_C04_011E": "unemployment_rate_black",
    "S2301_C04_012E": "unemployment_rate_native_american",
    "S2301_C04_013E": "unemployment_rate_asian",
    "S2301_C04_014E": "unemployment_rate_pacific_islander",
    "S2301_C04_015E": "unemployment_rate_other_race",
    "S2301_C04_016E": "unemployment_rate_two_or_more_races",
    "S2301_C04_017E": "unemployment_rate_hispanic",
    "S2301_C04_018E": "unemployment_rate_white_non_hispanic"
}

#Store processed data
all_data = []

#Loop through each file
for file in files:
    year = int(file[7:11])  # Extract year from filename
    path = os.path.join(folder_path, file)
    
    try:
        df = pd.read_csv(path, dtype=str)
        df = df[columns_map.keys()]
        df.rename(columns=columns_map, inplace=True)
        df["year"] = year
        
        # Remove commas and convert to numeric
        for col in df.columns:
            if col not in ["state", "year"]:
                df[col] = pd.to_numeric(df[col].str.replace(",", "", regex=False), errors="coerce")
        all_data.append(df)

    except Exception as e:
        print(f"Skipping {year} due to error: {e}")

#Merge and save
if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
     # Uppercase state names & remove metadata rows
    final_df["state"] = final_df["state"].str.strip().str.upper()
    final_df = final_df[final_df["state"].isin(us_states_upper)]
    cols = final_df.columns.tolist()
    cols.remove("year")
    cols.insert(1, "year")
    final_df = final_df[cols]

    # Save
    output_file = os.path.join(folder_path, "Employment_Cleaned_2010_2019.csv")
    final_df.to_csv(output_file, index=False)
    print(f"Final employment data saved to:\n{output_file}")

else:
    print("No valid employment data processed.")


Final employment data saved to:
C:/Users/mayan/Downloads/DJ/Study/USF/Spring 2025/Data Visualization/Final Project/Project/Employment Status\Employment_Cleaned_2010_2019.csv


In [28]:
# Display the cleaned employment data
employment_df = pd.read_csv("C:/Users/mayan/Downloads/DJ/Study/USF/Spring 2025/Data Visualization/Final Project/Project/Employment Status/Employment_Cleaned_2010_2019.csv")
employment_df

,state,year,population_16_plus,in_labor_force_16_plus,employed_16_plus,unemployment_rate_total,unemployment_rate_male,unemployment_rate_female,unemployment_rate_age_16_19,unemployment_rate_age_20_24,...,unemployment_rate_age_75_plus,unemployment_rate_white,unemployment_rate_black,unemployment_rate_native_american,unemployment_rate_asian,unemployment_rate_pacific_islander,unemployment_rate_other_race,unemployment_rate_two_or_more_races,unemployment_rate_hispanic,unemployment_rate_white_non_hispanic
0,ALABAMA,2010,3714504.0,60.5,54.8,8.7,7.5,8.1,28.3,15.9,...,4.7,6.7,14.6,10.8,5.6,0.5,8.6,12.7,8.7,6.6
1,ALASKA,2010,528189.0,72.0,62.9,8.6,8.8,6.6,23.0,14.8,...,4.5,6.3,8.9,21.7,4.7,11.3,9.9,14.1,9.0,6.3
2,ARIZONA,2010,4813496.0,62.2,57.1,7.7,7.2,6.6,22.1,11.5,...,7.3,6.8,11.6,16.8,6.4,6.3,9.4,12.3,9.3,6.2
3,ARKANSAS,2010,2247624.0,60.9,55.8,7.8,7.2,6.9,25.1,13.9,...,3.7,6.5,15.2,9.4,5.7,19.4,7.9,10.4,7.5,6.4
4,CALIFORNIA,2010,28445585.0,64.7,58.5,9.0,8.2,8.2,28.1,14.0,...,5.8,8.4,14.2,14.2,7.2,11.1,10.2,12.9,10.6,7.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,VIRGINIA,2019,6799060.0,65.9,61.1,4.6,3.8,4.2,16.6,9.1,...,2.9,2.5,2.0,3.9,7.3,6.2,3.3,6.9,5.1,6.7
496,WASHINGTON,2019,5941281.0,64.5,60.5,5.0,4.5,4.5,18.1,8.8,...,3.3,2.8,3.7,4.6,7.8,10.5,3.9,6.9,6.0,7.2
497,WEST VIRGINIA,2019,1491316.0,53.3,49.7,6.5,6.4,6.2,21.1,12.9,...,2.9,2.7,2.7,6.4,10.5,9.3,3.6,45.5,5.5,10.6
498,WISCONSIN,2019,4659582.0,66.5,64.0,3.6,3.0,3.3,10.0,6.1,...,2.4,2.2,2.5,3.0,10.0,9.2,3.6,4.3,5.3,8.3


## ✅ Employment Data (2010–2019) Preview Summary

This dataset captures U.S. state-level labor force and unemployment statistics from **2010 to 2019**.

- 📌 **500 rows** — representing 50 U.S. states × 10 years  
- 📊 **24 columns** covering:

  - 👥 Total population aged 16+  
  - 🛠️ Labor force participation and employment rates  
  - 📉 Unemployment rates overall, and segmented by:
    - 👨‍👩‍👧‍👦 Gender (male/female)  
    - 🗓️ Age groups (16–19, 20–24, ..., 75+)  
    - 🌎 Race/Ethnicity (White, Black, Hispanic, Asian, etc.)

---

This dataset is now clean, structured, and ready to support economic trend analysis or correlation studies with crime data.
with crime data.


## 💰 Step 6: Unzip Income Data Files (2010–2019)

In this step, the ZIP archive containing income data files for each year is extracted.

Only the `*-Data.csv` files (containing the actual data tables) are extracted — skipping metadata or notes.

The extracted files are stored in the **Income Data** directory for further cleaning and analysis.


In [33]:
# Define the ZIP file path and destination extraction folder
income_zip_path = r"C:/Users/mayan/Downloads/DJ/Study/USF/Spring 2025/Data Visualization/Final Project/Project/S1901 Income in the Past 12 Months.zip"
income_extract_path = r"C:/Users/mayan/Downloads/DJ/Study/USF/Spring 2025/Data Visualization/Final Project/Project/Income Data"

# Create folder if it doesn't already exist
os.makedirs(income_extract_path, exist_ok=True)

# Extract only the CSV data files (skip metadata and notes)
with zipfile.ZipFile(income_zip_path, 'r') as income_zip_ref:
    for file in income_zip_ref.namelist():
        if file.endswith("Data.csv"):   # Only extract files that end in "Data.csv"
            income_zip_ref.extract(file, income_extract_path)

print("Income ZIP extracted!")

Income ZIP extracted!


## 💵 Step 7: Clean and Combine Income Data (2010–2019)

This step extracts selected income metrics from each year’s CSV file between 2010 and 2019 and compiles them into a unified dataset.

The following metrics are retained:
- Total households
- Median and mean income for:
  - All households
  - Families
  - Married-couple families
  - Nonfamily households

The final dataset includes only the 50 U.S. states and omits additional territories for stan
dzation.

📁 Output: `Income_Cleaned_2010_2019.csv`


In [84]:
# Define the columns to keep and how to rename them
selected_columns = {
    'NAME': 'state',
    'S1901_C01_001E': 'total_households',
    'S1901_C01_012E': 'median_income_household',
    'S1901_C02_012E': 'median_income_family',
    'S1901_C03_012E': 'median_income_married',
    'S1901_C04_012E': 'median_income_nonfamily',
    'S1901_C01_013E': 'mean_income_household',
    'S1901_C02_013E': 'mean_income_family',
    'S1901_C03_013E': 'mean_income_married',
    'S1901_C04_013E': 'mean_income_nonfamily'
}
# Define official 50 U.S. states (uppercase for matching)
us_states = [
    "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut", "Delaware",
    "Florida", "Georgia", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa", "Kansas", "Kentucky",
    "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota", "Mississippi",
    "Missouri", "Montana", "Nebraska", "Nevada", "New Hampshire", "New Jersey", "New Mexico",
    "New York", "North Carolina", "North Dakota", "Ohio", "Oklahoma", "Oregon", "Pennsylvania",
    "Rhode Island", "South Carolina", "South Dakota", "Tennessee", "Texas", "Utah", "Vermont",
    "Virginia", "Washington", "West Virginia", "Wisconsin", "Wyoming"
]
us_states_upper = [s.upper() for s in us_states]

# Initialize container for all yearly data
all_data = []
folder_path = income_extract_path  # Path to extracted CSV files

# Loop through all extracted income data files
for file in os.listdir(folder_path):
    if file.endswith(".csv") and "Data" in file:
        try:
            year = int(file[7:11])  # Extract year from filename
            df = pd.read_csv(os.path.join(folder_path, file), dtype=str)
            
            # Remove metadata row
            df = df[df["NAME"].str.strip() != "Geographic Area Name"]
            df = df[df["NAME"].str.strip() != "Geography"]
            # Only keep selected columns and rename
            df = df[list(selected_columns.keys())].copy()
            df.rename(columns=selected_columns, inplace=True)
            df['year'] = year
            all_data.append(df)

        except Exception as e:
            print(f"Skipping {file} due to error: {e}")

# Combine all years into a single DataFrame
income_df = pd.concat(all_data, ignore_index=True)
# Clean the 'state' column
# Standardize column values and filter only valid states
income_df['state'] = income_df['state'].str.strip().str.upper()
valid_states = set(us_states_upper)
income_df = income_df[income_df['state'].isin(valid_states)]
income_df = income_df.reset_index(drop=True)
income_df


,state,total_households,median_income_household,median_income_family,median_income_married,median_income_nonfamily,mean_income_household,mean_income_family,mean_income_married,mean_income_nonfamily,year
0,ALABAMA,1821210,42081,52863,65188,22933,57655,68275,80927,33317,2010
1,ALASKA,248248,66521,77886,90821,42502,82091,93053,106686,53784,2010
2,ARIZONA,2326468,50448,59840,70143,33003,67436,77127,88273,45232,2010
3,ARKANSAS,1117154,39267,48491,57984,22039,53253,62497,72578,31538,2010
4,CALIFORNIA,12392852,60883,69322,82998,40588,83483,92942,107696,58117,2010
...,...,...,...,...,...,...,...,...,...,...,...
495,VIRGINIA,3151045,74222,90141,106477,44781,101746,118940,136615,63832,2019
496,WASHINGTON,2848396,73775,88660,102341,45823,98983,114940,129918,64637,2019
497,WEST VIRGINIA,732585,46711,59607,70774,25955,63680,75996,87040,38392,2019
498,WISCONSIN,2358156,61747,78679,90612,36528,80674,97431,110447,49088,2019


## 🧾 Step 8: Preview and Organize Cleaned Income Dataset

This step loads the final cleaned income dataset and performs a quick structural adjustment to improve readability1]`.
- Moves the `year` column directly after `state` for logical ordering.
- Displays the dataset to verify correctness and structure before further analysis.


In [87]:
# Reorder columns: move 'year' after 'state'
cols = income_df.columns.tolist()
cols.remove('year')
cols.insert(cols.index('state') + 1, 'year')
income_df = income_df[cols]

# Save the cleaned income data to CSV
output_path = os.path.join(folder_path, "Income_Cleaned_2010_2019.csv")
income_df.to_csv(output_path, index=False)

print(f"Final income data saved to:\n{output_path}")
income_df

Final income data saved to:
C:/Users/mayan/Downloads/DJ/Study/USF/Spring 2025/Data Visualization/Final Project/Project/Income Data\Income_Cleaned_2010_2019.csv


,state,year,total_households,median_income_household,median_income_family,median_income_married,median_income_nonfamily,mean_income_household,mean_income_family,mean_income_married,mean_income_nonfamily
0,ALABAMA,2010,1821210,42081,52863,65188,22933,57655,68275,80927,33317
1,ALASKA,2010,248248,66521,77886,90821,42502,82091,93053,106686,53784
2,ARIZONA,2010,2326468,50448,59840,70143,33003,67436,77127,88273,45232
3,ARKANSAS,2010,1117154,39267,48491,57984,22039,53253,62497,72578,31538
4,CALIFORNIA,2010,12392852,60883,69322,82998,40588,83483,92942,107696,58117
...,...,...,...,...,...,...,...,...,...,...,...
495,VIRGINIA,2019,3151045,74222,90141,106477,44781,101746,118940,136615,63832
496,WASHINGTON,2019,2848396,73775,88660,102341,45823,98983,114940,129918,64637
497,WEST VIRGINIA,2019,732585,46711,59607,70774,25955,63680,75996,87040,38392
498,WISCONSIN,2019,2358156,61747,78679,90612,36528,80674,97431,110447,49088


## 💸 Income Data (2010–2019) Preview Summary

The cleaned income dataset includes:

**📍 500 rows – representing all 50 U.S. states (excluding: Puerto Rico, D.C., etc.) across 10 years (2010–2019)**

**📊 11 columns capturing key income statistics:**

- `total_households`: Number of households per state-year  
- **Median income**: for households, families, married couples, nonfamily households  
- **Mean income**: for the same groups above  

✅ Data is clean, numeric, and chronologically aligned — ready for analysis and comparison with crime data.
